# Notebook 01: JAX Fundamentals

## Why This Matters

JAX is Google's answer to "what if NumPy ran on TPUs and could automatically differentiate itself?" It powers the majority of Google DeepMind's research, including Gemini's training infrastructure. Understanding JAX is now a prerequisite for roles involving large-scale ML research, TPU utilization, or the growing ecosystem of JAX-native models. Its design philosophy — functional purity, explicit state, composable transforms — also makes you a better programmer in any framework.

## Background

JAX compiles Python+NumPy code down to **XLA (Accelerated Linear Algebra)**, Google's domain-specific compiler for ML workloads. XLA can target CPU, GPU, and TPU. The key insight: rather than executing operations eagerly like standard NumPy, JAX traces your function, builds an intermediate representation (HLO — High Level Operations), and lets XLA optimize and compile it.

This is why the **first call is slow** (compilation) but subsequent calls are fast (running the compiled binary). The tradeoff: your functions must be **pure** — no Python side effects, no in-place mutation, no reading global state — so XLA can safely reason about them.

In [ ]:
%pip install -q jax jaxlib numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import time
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())
print("Ready.")

## 1. JAX Arrays vs NumPy Arrays

JAX arrays (`jnp.ndarray`) look like NumPy arrays but have two critical differences:

1. **Immutability**: JAX arrays cannot be modified in-place. `x[0] = 5` raises an error.
2. **Device placement**: JAX arrays live on a device (CPU/GPU/TPU). Data movement is explicit.

The immutability is not a limitation — it's what enables the functional transforms. If arrays could mutate, `jax.grad` couldn't safely trace through your code.

In [ ]:
# NumPy array — mutable
np_arr = np.array([1.0, 2.0, 3.0])
np_arr[0] = 99.0
print("NumPy in-place:", np_arr)

# JAX array — immutable
jax_arr = jnp.array([1.0, 2.0, 3.0])
print("JAX array:", jax_arr)
print("Type:", type(jax_arr))
print("Device:", jax_arr.devices())

# Attempting in-place mutation raises TypeError
try:
    jax_arr[0] = 99.0
except TypeError as e:
    print(f"JAX in-place error: {e}")

# Instead: use .at[].set() — returns a NEW array
jax_arr_modified = jax_arr.at[0].set(99.0)
print("Original (unchanged):", jax_arr)
print("New array:", jax_arr_modified)

## 2. Functional Purity: Why JAX Functions Must Have No Side Effects

A **pure function** always returns the same output for the same input and has no observable side effects (no printing, no writing files, no mutating global state).

JAX's transforms (`jit`, `grad`, `vmap`) work by **tracing** your function with abstract values — they need to call your function multiple times with symbolic inputs, so any side effect would fire unexpectedly. The rule:

> If JAX traces your function with `x = AbstractArray(shape=(3,), dtype=float32)`, a `print(x)` fires during tracing but NOT during actual execution. Your mental model must account for this.

**Pitfall**: `print()` inside `jit` runs once at trace time, not on every call. Use `jax.debug.print` for runtime debugging.

In [ ]:
# Demonstrating trace-time vs runtime behavior
import jax

counter = {"calls": 0}

def impure_fn(x):
    counter["calls"] += 1  # side effect!
    print(f"Python print fired (trace or eager): counter={counter['calls']}")
    return x * 2.0

# Eager (not jitted): fires every call
print("=== Eager calls ===")
impure_fn(jnp.array(3.0))
impure_fn(jnp.array(3.0))

counter["calls"] = 0

# JIT-compiled: Python code only fires at TRACE TIME
jit_fn = jax.jit(impure_fn)
print("
=== JIT calls ===")
_ = jit_fn(jnp.array(3.0))   # traces + compiles
_ = jit_fn(jnp.array(3.0))   # runs compiled binary, no Python
_ = jit_fn(jnp.array(3.0))
print(f"counter incremented {counter['calls']} time(s) despite 3 JIT calls")

# Correct approach: jax.debug.print (runs at runtime)
@jax.jit
def debug_fn(x):
    jax.debug.print("Runtime value of x: {x}", x=x)
    return x * 2.0

print("
=== jax.debug.print (fires every call) ===")
_ = debug_fn(jnp.array(3.0))
_ = debug_fn(jnp.array(5.0))

## 3. XLA Compilation Model

When you call a `jit`-compiled function, JAX:
1. **Traces** your Python function with abstract tracers (shape + dtype, no actual values)
2. Builds an **HLO (High Level Operations)** graph
3. Passes HLO to **XLA**, which performs fusion, layout optimization, and code generation
4. **Caches** the compiled binary keyed on (function, input shapes, input dtypes)

A new compilation happens if shapes change. This is why TPU training with variable-length sequences is painful — each unique length triggers recompilation. The standard fix: **pad to fixed shapes**.

In [ ]:
# Demonstrating compilation caching behavior
@jax.jit
def matrix_multiply(A, B):
    return jnp.dot(A, B)

# First call: includes compilation time
key = jax.random.PRNGKey(0)
A = jax.random.normal(key, (1000, 1000))
B = jax.random.normal(key, (1000, 1000))

# Warm up (trigger compilation)
_ = matrix_multiply(A, B).block_until_ready()

# Time the compiled version
t0 = time.time()
for _ in range(10):
    _ = matrix_multiply(A, B).block_until_ready()
jax_compiled_time = (time.time() - t0) / 10

print(f"JAX compiled matmul (1000x1000): {jax_compiled_time*1000:.2f} ms/call")

# If we change shape: recompilation
A2 = jax.random.normal(key, (2000, 2000))
B2 = jax.random.normal(key, (2000, 2000))
t0 = time.time()
_ = matrix_multiply(A2, B2).block_until_ready()  # recompiles for new shape
print(f"First call with new shape (includes recompilation): {(time.time()-t0)*1000:.2f} ms")

_ = matrix_multiply(A2, B2).block_until_ready()
t0 = time.time()
for _ in range(5):
    _ = matrix_multiply(A2, B2).block_until_ready()
print(f"Subsequent calls with same shape: {(time.time()-t0)/5*1000:.2f} ms/call")

## 4. JAX's Explicit PRNG System

NumPy uses a **global random state** (seeded once, advances implicitly). JAX uses **explicit PRNG keys** that must be threaded through your program manually.

Why? Global state is a side effect — it breaks functional purity and makes `jit`/`vmap` impossible to reason about. JAX's solution: treat randomness as **a pure function of a key**. Same key always gives same random values.

The **Threefry** hash-based PRNG also makes JAX trivially reproducible and parallelizable — you can split a key into two independent keys without any global coordination.

$$\text{key}_{\text{new}} = \text{split}(\text{key}_{\text{parent}})$$

In [ ]:
# JAX PRNG: explicit key management
key = jax.random.PRNGKey(42)
print("Initial key:", key)

# Generating random values: same key = same result
x1 = jax.random.normal(key, shape=(3,))
x2 = jax.random.normal(key, shape=(3,))
print("Same key produces same values:")
print("  x1:", x1)
print("  x2:", x2)
print("  Equal:", jnp.allclose(x1, x2))

# Splitting keys: the correct pattern
key, subkey = jax.random.split(key)
y1 = jax.random.normal(subkey, shape=(3,))
print("
After split, new values:", y1)

# Multiple splits at once
keys = jax.random.split(key, num=4)
print("
4 independent keys from one split:")
for i, k in enumerate(keys):
    print(f"  key[{i}]: {k} → sample: {jax.random.normal(k, shape=(2,))}")

# Common distributions
key, k1, k2, k3, k4 = jax.random.split(key, 5)
print("
Common distributions:")
print("  Normal:    ", jax.random.normal(k1, shape=(4,)))
print("  Uniform:   ", jax.random.uniform(k2, shape=(4,)))
print("  Bernoulli: ", jax.random.bernoulli(k3, p=0.7, shape=(4,)))
print("  Categorical:", jax.random.categorical(k4, logits=jnp.array([1.0, 2.0, 3.0]), shape=(4,)))

## 5. JAX vs NumPy Speedup: Matrix Multiplication Benchmark

JAX's JIT-compiled operations, especially on GPU/TPU, dramatically outperform CPU NumPy for large matrix operations. Even on CPU, XLA's fusion and BLAS integration can match or beat NumPy. The real speedup comes from hardware acceleration.

Below we benchmark on whatever backend is available. On CPU you'll see modest improvement; on GPU/TPU the gap widens dramatically.

In [ ]:
# Benchmark: JAX vs NumPy matmul at increasing sizes
sizes = [128, 256, 512, 1024, 2048]
np_times = []
jax_times = []

@jax.jit
def jax_matmul(A, B):
    return jnp.dot(A, B)

for n in sizes:
    rng = np.random.default_rng(0)
    A_np = rng.standard_normal((n, n)).astype(np.float32)
    B_np = rng.standard_normal((n, n)).astype(np.float32)
    A_jax = jnp.array(A_np)
    B_jax = jnp.array(B_np)

    # Warm up JAX
    _ = jax_matmul(A_jax, B_jax).block_until_ready()

    # Time NumPy
    t0 = time.perf_counter()
    for _ in range(5):
        _ = np.dot(A_np, B_np)
    np_times.append((time.perf_counter() - t0) / 5 * 1000)

    # Time JAX
    t0 = time.perf_counter()
    for _ in range(5):
        _ = jax_matmul(A_jax, B_jax).block_until_ready()
    jax_times.append((time.perf_counter() - t0) / 5 * 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(sizes, np_times, 'o-', label='NumPy', color='#e74c3c')
ax1.plot(sizes, jax_times, 's-', label=f'JAX ({jax.default_backend()})', color='#2ecc71')
ax1.set_xlabel('Matrix size N (NxN)')
ax1.set_ylabel('Time (ms)')
ax1.set_title('MatMul Time: NumPy vs JAX')
ax1.legend()
ax1.grid(True, alpha=0.3)

speedups = [np_t / jax_t for np_t, jax_t in zip(np_times, jax_times)]
ax2.bar(range(len(sizes)), speedups, color='#3498db', alpha=0.8)
ax2.set_xticks(range(len(sizes)))
ax2.set_xticklabels([str(s) for s in sizes])
ax2.set_xlabel('Matrix size N')
ax2.set_ylabel('Speedup (NumPy time / JAX time)')
ax2.set_title(f'JAX Speedup vs NumPy ({jax.default_backend()})')
ax2.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='No speedup')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('jax_vs_numpy_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()

print("
Results summary:")
print(f"{'Size':>6} | {'NumPy (ms)':>10} | {'JAX (ms)':>10} | {'Speedup':>8}")
print("-" * 45)
for n, np_t, jax_t, sp in zip(sizes, np_times, jax_times, speedups):
    print(f"{n:>6} | {np_t:>10.2f} | {jax_t:>10.2f} | {sp:>8.2f}x")

## Summary

### Key Concepts

| Concept | Description | Key Gotcha |
|---------|-------------|------------|
| JAX arrays | Immutable, device-placed NumPy-compatible arrays | Use `.at[].set()` for updates |
| Functional purity | Functions must be pure: no side effects | `print()` fires at trace time, not runtime |
| XLA compilation | First call compiles, subsequent calls run the binary | Shape changes trigger recompilation |
| PRNG keys | Explicit threaded keys, no global state | Always `split()` before generating |
| `block_until_ready()` | JAX is async — call this when benchmarking | Without it, you time dispatch, not compute |

**Next**: `02_grad_vmap_jit.ipynb` — the three transforms that make JAX transformative.